# YOLOv8 Inference and ONNX Export

This notebook prepares the best YOLOv8 detector for deployment. It downloads the selected PyTorch checkpoint, exports the model to ONNX with preprocessing-compatible input dimensions and embedded non-maximum suppression, and validates the exported graph against the TACO validation split.

The exported ONNX model can be executed with ONNX Runtime in an application container that does not include PyTorch or Ultralytics. The accompanying `class_names.json` file preserves the mapping from numeric prediction IDs to the reduced TACO class names.

## Install Packages

Install the libraries required to load the training checkpoint, export it to ONNX, and evaluate the exported model. Ultralytics performs the conversion and may install ONNX-specific export dependencies when they are not already available in the runtime. These training-side dependencies are only needed during export; the final deployment container can use ONNX Runtime without PyTorch.

In [ ]:
!pip install torchmetrics pycocotools gdown tqdm tensorboard albumentations ultralytics

## Download the Best Model

Download the `best.pt` checkpoint selected by validation performance during training. This checkpoint corresponds to the best epoch rather than the final epoch, which avoids deploying the lower-performing weights produced after the validation metric had plateaued.

The downloaded file is stored as `./best.pt` and becomes the source artifact for ONNX export. In a production workflow, the file should be versioned together with its taxonomy, image size, and validation metrics.

In [ ]:
!gdown 19q4FulQTz2rot074mA8R537CAREDtRz1 

## Export to ONNX

Load the PyTorch checkpoint and convert it into a fixed-shape ONNX graph accepting one `1024 × 1024` image at a time. Graph simplification improves runtime compatibility, while embedded non-maximum suppression returns final detections instead of the model's raw candidate boxes. The export also writes `class_names.json`, which must be deployed with the ONNX file.

The configured confidence threshold (`0.25`), IoU threshold (`0.7`), and maximum detection count (`300`) become part of the exported postprocessing behavior when NMS is embedded. Predictions below the export confidence threshold cannot be recovered later by passing a lower threshold to ONNX Runtime. Export with a lower confidence threshold or without embedded NMS if the serving application must tune this value dynamically.

In [ ]:
import json
from ultralytics import YOLO

checkpoint = "./best.pt"

model = YOLO(checkpoint)

onnx_path = model.export(
    format="onnx",
    imgsz=1024,
    batch=1,
    dynamic=False,
    simplify=True,
    opset=17,
    nms=True,       # include NMS in the exported graph
    conf=0.25,      # production threshold, tune if needed
    iou=0.7,
    max_det=300,
)

with open("class_names.json", "w") as file:
    json.dump(model.names, file)

print(onnx_path)

Ultralytics 8.4.88 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 93 layers, 25,849,603 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/taco_yolo_runs/yolov8m_continued_42/weights/best.pt' with input shape (1, 3, 1024, 1024) BCHW and output shape(s) (1, 300, 6) (49.6 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 234ms
Prepared 4 packages in 1.88s
Installed 4 packages in 257ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.27.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 2.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.2

## Validate the Exported Model

Reload the ONNX artifact through Ultralytics and run it on the same validation split used for the final PyTorch checkpoint. This checks that conversion succeeded, all operators are supported, and the exported model still produces reasonable bounding-box metrics before it is copied into the deployment image.

Use the same image size, batch size, IoU threshold, and maximum detection count as the reference validation whenever metric parity is required. Because this graph was exported with embedded NMS at confidence `0.25`, setting `conf=0.001` in this validation call cannot restore candidates already removed inside the graph. A strict PyTorch-versus-ONNX comparison therefore requires exporting with `conf=0.001` or exporting without NMS and performing postprocessing outside the graph.

In [ ]:
onnx_model = YOLO(onnx_path)

yaml_path = "/content/taco_yolo/taco.yaml"
metrics = onnx_model.val(
    data=str(yaml_path),
    imgsz=1024,
    batch=12,
    conf=0.001,
    iou=0.7,
    max_det=300,
)

print(metrics.box.map50, metrics.box.map)

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Ultralytics 8.4.88 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Loading /content/drive/MyDrive/taco_yolo_runs/yolov8m_continued_42/weights/best.onnx for ONNX Runtime inference...
requirements: Ultralytics requirement ['onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 5 packages in 155ms
Prepared 1 package in 2.04s
Installed 1 package in 70ms
 + onnxruntime-gpu==1.27.0

requirements: AutoUpdate success ✅ 2.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

WARNING ⚠️ CUDA requested but CUDAExecutionProvider not available. Using CPU...
Using ONNX Runtime 1.27.0 with CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 1024, 1024)
val: Fast image access ✅ (ping: 0.0±0.0 ms, 